# LLM-Lab on Colab (quickstart)

A thin wrapper to run the project on a Colab GPU while **persisting artifacts to Google Drive**.

Strategy: train against fast **local** disk and mirror the run directory to Drive
periodically (and at the end), so the training loop never pays Drive latency.

1. Mount Drive
2. Configure + install
3. (once) Prepare data
4. Train (with a background sync to Drive)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
from pathlib import Path

# --- configuration ---------------------------------------------------------
PROJECT_PATH = Path("/content/drive/MyDrive/LLM-Lab")   # repo (persistent, on Drive)
LOCAL_RUNS   = Path("/content/runs")                    # fast, ephemeral artifacts
CONFIG       = "configs/experiments/scaling_model/xl3.yaml"

os.chdir(PROJECT_PATH)
print("cwd:   ", Path.cwd())
print("config:", CONFIG)

# --- install (editable) + device check -------------------------------------
!pip install -q -e . matplotlib
!python -c "import torch; print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')"

In [ ]:
# Optional: prepare the dataset + tokenizer once (skipped if already present).
import subprocess
import sys
from llm_lab.config import load_config

processed = Path(load_config(CONFIG)["data"]["processed_dir"])
if (processed / "train.pt").exists():
    print("data already prepared:", processed)
else:
    subprocess.run(
        [sys.executable, "scripts/prepare_data.py", "--config", CONFIG], check=True
    )

In [ ]:
import threading

from llm_lab.config import load_config
from llm_lab.utils.sync import sync_tree

cfg = load_config(CONFIG)
name = f"{cfg['experiment']}/{cfg['run_name']}"
LOCAL_RUN = LOCAL_RUNS / name
DRIVE_RUN = PROJECT_PATH / "artifacts" / "runs" / name

# Resume across sessions: pull the run down from Drive if it is not local yet.
if not LOCAL_RUN.exists() and DRIVE_RUN.exists():
    print("restoring from Drive:", sync_tree(DRIVE_RUN, LOCAL_RUN))

# Background mirror to Drive every 10 minutes, so a disconnect loses little.
stop = threading.Event()

def _sync_loop():
    while not stop.wait(600):
        try:
            print("[sync]", sync_tree(LOCAL_RUN, DRIVE_RUN), flush=True)
        except Exception as exc:  # keep training even if a sync hiccups
            print("[sync] failed:", exc, flush=True)

threading.Thread(target=_sync_loop, daemon=True).start()

# Train against fast local disk (resume is automatic if a checkpoint exists).
!python scripts/train.py --config {CONFIG} --checkpoint-dir {LOCAL_RUNS}

# Final sync.
stop.set()
print("[sync]", sync_tree(LOCAL_RUN, DRIVE_RUN))